# Pantera entrance v1 — Kaggle GPU render (chunked)

Renders 1500 frames of Pantera's Western Heart entrance video with **Blender 4.0.2 Cycles on a Kaggle T4 GPU**, in **250-frame chunks** (one notebook run per chunk).

**Before running:** Settings (right panel) → **Accelerator: GPU T4 x2**, **Internet: ON**, Persistence: *not needed*.
Set `CHUNK` below to the chunk number (0–5) for this run, then **Run All**.

Each run: checks GPU → installs Blender 4.0.2 → downloads the v3 repaired model + build script + titantron (SHA-256 verified) → renders 250 frames with CUDA → zips the chunk for download from `/kaggle/working/`.

Model: `pantera_rigged_working_v3.blend` — 18/18 mesh weight hashes == repaired v3 GLB (`5360d8cc…`, armpit de-bleed fix, gate 8/8 PASS). Scene: `build_v1.py` (Blender scene v4, procedural animation, 960×540 @16spp Cycles + denoise; post upscales to 1920×1080).


In [ ]:
# === CONFIG: set CHUNK 0..5 for this run ===
CHUNK = 0
SAMPLES = 16  # Cycles samples/px (960x540 + denoise); lower to 12 if a chunk is too slow
CHUNKS = {0: (0, 249), 1: (250, 499), 2: (500, 749), 3: (750, 999), 4: (1000, 1249), 5: (1250, 1499)}
assert CHUNK in CHUNKS, f"CHUNK must be 0-5, got {CHUNK}"
F0, F1 = CHUNKS[CHUNK]
print(f"chunk {CHUNK}: frames {F0}-{F1}, {SAMPLES}spp")


In [ ]:
# === GPU check ===
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                    "--format=csv"], capture_output=True, text=True)
print(r.stdout.strip() or "NO GPU VISIBLE — enable Accelerator: GPU in Settings!")
assert "Tesla T4" in r.stdout, "Expected a Tesla T4"


In [ ]:
# === Blender 4.0.2 portable + system deps ===
import os, subprocess, shutil
WORK = "/kaggle/working/pantera"
BLDIR = f"{WORK}/blender-4.0.2"
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f"{BLDIR}/blender"):
    print("installing system deps...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq",
                    "libgl1", "libxi6", "libxkbcommon0", "libsm6",
                    "libxrender1", "libxxf86vm1"], check=False)
    print("downloading Blender 4.0.2...")
    subprocess.run(["wget", "-q", "-O", f"{WORK}/blender.tar.xz",
                    "https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz"], check=True)
    subprocess.run(["tar", "-xf", f"{WORK}/blender.tar.xz", "-C", WORK], check=True)
    os.remove(f"{WORK}/blender.tar.xz")
r = subprocess.run([f"{BLDIR}/blender", "--version"], capture_output=True, text=True)
print(r.stdout.strip().splitlines()[0])


In [ ]:
# === Download render assets (SHA-256 verified) ===
import hashlib, os, urllib.request
RAW = "https://raw.githubusercontent.com/mhvnsnt/western-heart/muse/wh-pantera-entrance-v1/promo/pantera-entrance-v1"
ASSETS = {
    "kit/pantera_rigged_working_v3.blend":
        ("https://raw.githubusercontent.com/mhvnsnt/western-heart/muse/wh-pantera-entrance-v1/promo/pantera-entrance-v1/kaggle/pantera_rigged_working_v3.blend", "a5d4adb3076e73feb65dad826e235e9c9b26a23f3913507304f33078783d5934"),
    "kit/build_v1.py":
        ("https://raw.githubusercontent.com/mhvnsnt/western-heart/muse/wh-pantera-entrance-v1/promo/pantera-entrance-v1/build/build_v1.py", "c15eb760337b69415002dae6aeeb65b3a7d9032098b6c68c550d8b430238d321"),
    "kit/assets/titantron_plate_1920.png":
        ("https://raw.githubusercontent.com/mhvnsnt/western-heart/muse/wh-pantera-entrance-v1/promo/pantera-entrance-v1/assets/titantron_plate_1920.png", "de78d7a3c73c75d272a6971c0232b790e73f4f5fa49e4cefc49bc8bf7180babb"),
}
WORK = "/kaggle/working/pantera"
for rel, (url, sha) in ASSETS.items():
    dst = f"{WORK}/{rel}"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst):
        h = hashlib.sha256(open(dst, "rb").read()).hexdigest()
        if h == sha:
            print(f"cached OK: {rel}")
            continue
        print(f"re-downloading (hash mismatch): {rel}")
    print(f"downloading {rel}...")
    urllib.request.urlretrieve(url, dst)
    h = hashlib.sha256(open(dst, "rb").read()).hexdigest()
    assert h == sha, f"SHA256 MISMATCH for {rel}: {h}"
    print(f"OK {rel} ({os.path.getsize(dst)} bytes)")


In [ ]:
# === RENDER the chunk (Blender headless, Cycles CUDA) ===
import os, subprocess, time
WORK = "/kaggle/working/pantera"
env = dict(os.environ)
env.update({
    "PANTERA_KIT": f"{WORK}/kit",
    "PANTERA_BLEND": f"{WORK}/kit/pantera_rigged_working_v3.blend",
    "PANTERA_TRON": f"{WORK}/kit/assets/titantron_plate_1920.png",
    "PANTERA_FRAMES": f"{WORK}/frames",
    "PANTERA_DEVICE": "CUDA",
    "PANTERA_SAMPLES": str(SAMPLES),
    "PANTERA_F0": str(F0),
    "PANTERA_F1": str(F1),
})
t0 = time.time()
p = subprocess.Popen(
    [f"{WORK}/blender-4.0.2/blender", "-b", "-P", f"{WORK}/kit/build_v1.py",
     "--", "--mode", "full"],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    s = line.strip()
    if any(k in s for k in ("frame ", "CUDA devices", "rendering frames",
                            "tron texture", "baking ", "DONE", "Error", "Traceback")):
        print(s, flush=True)
p.wait()
mins = (time.time() - t0) / 60
print(f"blender exit={p.returncode}, wall time={mins:.1f} min")
assert p.returncode == 0, "blender render failed"


In [ ]:
# === Verify chunk: count frames, show a sample ===
import os, glob
from IPython.display import Image, display
WORK = "/kaggle/working/pantera"
frames = sorted(glob.glob(f"{WORK}/frames/f*.png"))
print(f"frames on disk: {len(frames)} (expect {F1 - F0 + 1})")
assert len(frames) == F1 - F0 + 1, "FRAME COUNT MISMATCH"
print("first:", os.path.basename(frames[0]), "last:", os.path.basename(frames[-1]))
mid = frames[len(frames) // 2]
print("sample frame:", os.path.basename(mid))
display(Image(filename=mid, width=640))


In [ ]:
# === Zip chunk for download ===
import hashlib, os, subprocess, time
WORK = "/kaggle/working/pantera"
ZIP = f"/kaggle/working/pantera_chunk{CHUNK:02d}_f{F0:05d}-{F1:05d}.zip"
if os.path.exists(ZIP):
    os.remove(ZIP)
subprocess.run(["zip", "-q", "-j", ZIP] +
               [f"{WORK}/frames/f{f:05d}.png" for f in range(F0, F1 + 1)],
               check=True)
h = hashlib.sha256(open(ZIP, "rb").read()).hexdigest()
print(f"{os.path.basename(ZIP)}  {os.path.getsize(ZIP)} bytes")
print(f"sha256: {h}")
print("Download it from the notebook's right-panel file viewer (/kaggle/working/).")
